In [1]:
import cv2
import torch
import torch.nn.functional as F
import numpy as np
import pickle
import csv
import os
import datetime
from collections import Counter
from PIL import Image
from torchvision import transforms
from facenet_pytorch import MTCNN, InceptionResnetV1

/home/shk/Documents/projects/python/shkreco/face_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ──────────────────────────────────────────
# الإعداد
# ──────────────────────────────────────────

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"الجهاز: {device}")

# MTCNN للكشف والمحاذاة
mtcnn = MTCNN(
    image_size=160,
    margin=20,
    keep_all=False,
    device=device
)

# تحميل النموذج المدرَّب
backbone = InceptionResnetV1(pretrained='vggface2').to(device)
ckpt_path = 'checkpoints/arcface_best.pth'

if os.path.exists(ckpt_path):
    checkpoint = torch.load(ckpt_path, map_location=device)
    backbone.load_state_dict(checkpoint['backbone'])
    print(f"النموذج محمّل — best loss: {checkpoint['loss']:.4f}")
else:
    print("تحذير: لم يُوجد checkpoint — يستخدم pretrained فقط")

backbone.eval()

# تحميل قاعدة البيانات
with open('worker_db_local.pkl', 'rb') as f:
    worker_db = pickle.load(f)

names        = list(worker_db.keys())
db_matrix    = torch.tensor(
    np.array(list(worker_db.values())),
    dtype=torch.float32, device=device
)
db_matrix    = F.normalize(db_matrix, p=2, dim=1)
print(f"الأشخاص المسجلون: {names}\n")

# Transform — يطابق ما استخدمناه في التدريب
transform = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

الجهاز: cpu
النموذج محمّل — best loss: 0.1528
الأشخاص المسجلون: ['ismail']



In [3]:
# ──────────────────────────────────────────
# وظائف مساعدة
# ──────────────────────────────────────────

def apply_clahe(frame):
    """
    تحسين الإضاءة غير المتجانسة — مهم لبيئة المصنع.
    CLAHE = Contrast Limited Adaptive Histogram Equalization
    """
    lab        = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b    = cv2.split(lab)
    clahe      = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l          = clahe.apply(l)
    lab        = cv2.merge((l, a, b))
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)


def get_embedding(face_img_rgb):
    """استخراج embedding من صورة وجه RGB"""
    tensor = transform(Image.fromarray(face_img_rgb))
    tensor = tensor.unsqueeze(0).to(device)
    with torch.no_grad():
        emb = backbone(tensor)
    return F.normalize(emb, p=2, dim=1)


def identify(embedding, threshold=0.55):
    """مقارنة embedding مع قاعدة البيانات بـ cosine similarity"""
    sims     = torch.mm(embedding, db_matrix.T).squeeze(0)
    best_idx = torch.argmax(sims).item()
    best_sim = sims[best_idx].item()

    if best_sim >= threshold:
        return names[best_idx], best_sim
    return "Unknown", best_sim


# سجل الحضور — يمنع التكرار
attendance_log = {}

def record_attendance(name, cooldown_minutes=30):
    """تسجيل الحضور في CSV مع منع التكرار"""
    now = datetime.datetime.now()

    if name in attendance_log:
        elapsed = (now - attendance_log[name]).seconds / 60
        if elapsed < cooldown_minutes:
            return False  # سُجّل مؤخراً

    attendance_log[name] = now

    csv_file   = 'attendance.csv'
    file_exists = os.path.exists(csv_file)

    with open(csv_file, 'a', newline='') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(['الاسم', 'التاريخ', 'وقت الدخول'])
        writer.writerow([
            name,
            now.strftime('%Y-%m-%d'),
            now.strftime('%H:%M:%S'),
        ])

    return True

In [4]:
# ──────────────────────────────────────────
# حلقة التعرف الرئيسية
# ──────────────────────────────────────────

VOTE_WINDOW = 8       # عدد الإطارات قبل اتخاذ القرار
THRESHOLD   = 0.55    # حد الـ cosine similarity

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH,  640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

if not cap.isOpened():
    raise RuntimeError("لا يمكن فتح الكاميرا")

vote_buffer     = []   # تجميع نتائج الإطارات
current_label   = ""
current_color   = (200, 200, 200)

print("النظام يعمل — اضغط Q للخروج\n")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # 1. تحسين الإضاءة
    frame_processed = apply_clahe(frame)
    rgb             = cv2.cvtColor(frame_processed, cv2.COLOR_BGR2RGB)
    pil_img         = Image.fromarray(rgb)

    # 2. كشف الوجه
    boxes, _ = mtcnn.detect(pil_img)

    if boxes is not None and len(boxes) > 0:
        box        = boxes[0]   # أكبر وجه فقط
        h, w       = frame.shape[:2]
        x1, y1, x2, y2 = [int(v) for v in box]
        x1, y1     = max(0, x1), max(0, y1)
        x2, y2     = min(w, x2), min(h, y2)

        face_rgb = rgb[y1:y2, x1:x2]
        if face_rgb.size > 0:

            # 3. استخراج الـ embedding والتعرف
            emb        = get_embedding(face_rgb)
            name, sim  = identify(emb, THRESHOLD)
            vote_buffer.append(name)

            # 4. Voting — قرار بعد VOTE_WINDOW إطار
            if len(vote_buffer) >= VOTE_WINDOW:
                winner, count = Counter(vote_buffer).most_common(1)[0]

                # يجب أن يحصل على 75% من الأصوات
                if count >= VOTE_WINDOW * 0.75:
                    current_label = winner
                    if winner != "Unknown":
                        current_color = (0, 255, 0)
                        # 5. تسجيل الحضور
                        if record_attendance(winner):
                            print(f"سُجّل حضور: {winner} "
                                  f"— {datetime.datetime.now().strftime('%H:%M:%S')}")
                    else:
                        current_color = (0, 0, 255)
                else:
                    current_label = "غير واضح"
                    current_color = (0, 165, 255)

                vote_buffer.clear()

            # رسم المستطيل والاسم
            cv2.rectangle(frame, (x1, y1), (x2, y2),
                          current_color, 2)
            label_text = f"{current_label} ({sim:.2f})"
            cv2.putText(frame, label_text,
                        (x1, max(20, y1 - 10)),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.65, current_color, 2)

            # شريط تقدم الـ Voting
            progress = int((len(vote_buffer) / VOTE_WINDOW) * (x2 - x1))
            cv2.rectangle(frame,
                          (x1, y2 + 5),
                          (x1 + progress, y2 + 12),
                          (255, 200, 0), -1)

    # معلومات على الشاشة
    recorded = len(attendance_log)
    cv2.putText(frame,
                f"مسجّلون اليوم: {recorded} | Q للخروج",
                (10, 25),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                (255, 255, 255), 2)

    cv2.imshow("ENIE Attendance System", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

print(f"\nانتهت الجلسة — {len(attendance_log)} شخص مسجّل")
print("الحضور محفوظ في: attendance.csv")

النظام يعمل — اضغط Q للخروج



[W NNPACK.cpp:64] Could not initialize NNPACK! Reason: Unsupported hardware.
QFontDatabase: Cannot find font directory /home/shk/Documents/projects/python/shkreco/face_venv/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/shk/Documents/projects/python/shkreco/face_venv/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/shk/Documents/projects/python/shkreco/face_venv/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/shk/Documents/projects/python/shkreco/face_venv/lib/python3.11/site-packages/cv2/qt/fo


انتهت الجلسة — 0 شخص مسجّل
الحضور محفوظ في: attendance.csv
